# Hybrid Graph Part 2: Lexical-Graph Indexing & Semantic Query

This is Part 2 of the hybrid graph pipeline. It demonstrates how to:

1. Load LlamaIndex Documents from Part 1 (structured data with embedded lineage)
2. Index them into lexical-graph (Neptune graph + OpenSearch vectors)
3. Query with natural language (semantic search)
4. Correlate results back to the original typed nodes in document-graph

## Why This Matters

After Part 1, we have:
- Typed nodes in Neptune (queryable by Cypher: exact match, traversal)
- LlamaIndex Documents with lineage (ready for semantic indexing)

Part 2 adds:
- Semantic search capability (ask questions in natural language)
- Entity extraction and linking (lexical-graph's knowledge graph)
- Lineage-based correlation (trace semantic results → structured nodes)

## Data Flow

```
LlamaIndex Documents (from Part 1)
    ↓ lexical-graph extract_and_build()
Neptune: Entity graph + Chunk graph
OpenSearch: Vector embeddings
    ↓ semantic query
Query results with text + metadata
    ↓ lineage extraction
Original document-graph node in Neptune
```

## The Hybrid Advantage

| Query Type | Engine | Example |
|------------|--------|--------|
| Exact match | document-graph (Cypher) | "Find all papers by author X" |
| Traversal | document-graph (Cypher) | "Papers that cite paper Y" |
| Semantic | lexical-graph (vectors) | "Research about machine learning" |
| Hybrid | Both + lineage | "Find structured node for this semantic result" |

**Prerequisites:**
- `pip install graphrag-toolkit-lexical-graph`
- `pip install graphrag-toolkit-document-graph[graphrag]`
- Neptune cluster + OpenSearch Serverless collection (see 00-Setup)
- Part 1 completed (documents saved to `output/hybrid_foundation/`)


In [ ]:
# Setup
import json, os, pickle, re
from pathlib import Path
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory, VectorStoreFactory
from graphrag_toolkit.lexical_graph import LexicalGraphIndex, LexicalGraphQueryEngine

GRAPH_STORE = os.environ.get(
    'GRAPH_STORE',
    'neptune-db://<your-neptune-cluster-endpoint>:8182'
)
VECTOR_STORE = os.environ.get(
    'VECTOR_STORE',
    'aoss://<your-opensearch-serverless-endpoint>'
)
TENANT = 'hybrid_demo'

graph_store = GraphStoreFactory.for_graph_store(GRAPH_STORE)
vector_store = VectorStoreFactory.for_vector_store(VECTOR_STORE)
print('Stores connected')


## Step 1: Load Documents from Part 1

Load the LlamaIndex Documents that Part 1 created from structured data.
Each document contains the node's properties as text plus a lineage header
(`[NodeType | node_id | tenant]`) that will survive chunking.


In [ ]:
foundation_dir = Path('output/hybrid_foundation')

with open(foundation_dir / 'lexical_documents.pkl', 'rb') as f:
    lexical_documents = pickle.load(f)

print(f'Loaded {len(lexical_documents)} documents')
print(f'Sample: {lexical_documents[0].text[:150]}...')


## Step 2: Index into Lexical-Graph

`LexicalGraphIndex.extract_and_build()` does three things:

1. **Extract** — LLM-based entity and relationship extraction from the document text
2. **Build graph** — Write extracted entities and relationships to Neptune
3. **Build vectors** — Embed text chunks and store in OpenSearch for semantic search

After this step, you can query the data with natural language.


In [ ]:
# Index into lexical-graph (batch writes enabled — Neptune 1.4.7.0 supports it)
print(f"Indexing {len(lexical_documents)} documents (batch writes ON)...")

graph_index = LexicalGraphIndex(graph_store, vector_store)
graph_index.extract_and_build(lexical_documents, show_progress=True)

print("Indexing complete")


## Step 3: Query with Semantic Search

Use `LexicalGraphQueryEngine` to ask natural language questions.
The engine combines vector similarity search (OpenSearch) with graph traversal (Neptune)
to find relevant answers.

Behind the scenes:
1. Query is embedded → similar chunks found in OpenSearch
2. Chunks are expanded via graph traversal → related entities included
3. Context is assembled and returned (or passed to an LLM for generation)


In [ ]:
query_engine = LexicalGraphQueryEngine.for_traversal_based_search(graph_store, vector_store)

queries = [
    'What research discusses machine learning?',
    'Who are the authors working on graph databases?',
    'What citations exist between papers?',
]

for q in queries:
    print(f'\nQuery: {q}')
    results = query_engine.retrieve(q)
    if isinstance(results, list):
        print(f'  Results: {len(results)}')
        for i, r in enumerate(results[:2], 1):
            text = getattr(r.node, 'text', str(r))[:150]
            print(f'  {i}. {text}...')
    else:
        print(f'  Response: {str(results)[:150]}')


## Step 4: Correlate Back to Document-Graph (Lineage)

**This is the hybrid graph payoff.**

Semantic search returns text chunks. But we need to know: *which structured node did this come from?*

The lineage header (`[ResearchPaper | paper-001 | hybrid_demo]`) embedded in Part 1 tells us.
We extract it from the result text, then query Neptune directly for the original typed node
with all its structured properties.

This completes the loop:
- Natural language query → semantic result → lineage → typed Neptune node → structured data


In [ ]:
# Correlate: query results → Source nodes → document-graph
gs_direct = GraphStoreFactory.for_graph_store(GRAPH_STORE).__enter__()

q = 'What research discusses machine learning?'
results = query_engine.retrieve(q)
print(f'Query: {q}')
print(f'Results: {len(results)}\n')

for i, r in enumerate(results[:5], 1):
    text = getattr(r.node, 'text', str(r))
    meta = getattr(r.node, 'metadata', {})

    # Method 1: Parse lineage from text header [Type | node_id | tenant]
    match = re.search(r'\[(\w+) \| ([\w-]+) \| (\w+)\]', text)
    if match:
        ntype, node_id, tenant = match.group(1), match.group(2), match.group(3)
        label = f'__{ntype}__{tenant}__'
        original = gs_direct.execute_query(f"MATCH (n:`{label}`) WHERE n.id = '{node_id}' RETURN properties(n) as props LIMIT 1")
        print(f'{i}. [{ntype}] node_id={node_id}')
        if original:
            props = original[0].get('props', {})
            display = props.get('name', props.get('title', props.get('description', 'N/A')))
            print(f'   Neptune: {display}')
        else:
            print(f'   (not in Neptune)')
    else:
        # Method 2: Check Source nodes in lexical-graph
        sources = gs_direct.execute_query('MATCH (s:`__Source__`) WHERE s.node_type IS NOT NULL RETURN s.node_type as ntype, s.node_id as nid, s.tenant as t LIMIT 5')
        if sources and i == 1:
            print(f'{i}. [Source nodes available]:')  
            for s in sources[:3]:
                print(f'   {s}')
        else:
            # Method 3: Entity correlation fallback
            ec = meta.get('entity_contexts', {}).get('contexts', [])
            entities = [ent.get('entity', {}).get('value', '') for ctx in ec[:3] for ent in ctx.get('entities', [])]
            print(f'{i}. [entities: {", ".join(entities[:3])}]')
    print()

print('Lineage correlation complete')


## Summary

The hybrid pipeline:
1. **Document-graph** stores structured data as typed nodes in Neptune
2. **Lexical-graph** indexes the same data for semantic search (OpenSearch vectors)
3. **Lineage** embedded in text `[source_type | node_id | tenant]` survives chunking
4. **Correlation** extracts lineage from query results → fetches original node from Neptune

Same graph database (Neptune) holds both the document-graph nodes AND the lexical-graph structure.
